## Build BioMistral Medical RAG chatbot using bioMistral open source LLM

## Load the google drive


In [ ]:
from  google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Installation

In [ ]:
!pip install langchain sentence-transformers chromadb llama-cpp-python langchain_community pypdf python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 MB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.0/604.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━

# Importing Libraries

In [ ]:

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA,LLMChain

# Import the document

In [ ]:
loader = PyPDFDirectoryLoader("/content/drive/MyDrive/BioMistral/Data/")
docs = loader.load()

In [ ]:
len(docs) #number of pages
docs[6]

Document(metadata={'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf', 'page': 6}, page_content='2\nThese facts may seem frightening, but they need not be. The good\nnews is that you have a lot of power to protect and improve yourheart health. This guidebook will help you find out your own riskof heart disease and take steps to prevent it.\n“But,” you may still be thinking, “I take pretty good care of myself.\nI’m unlikely to get heart disease.” Yet a recent national survey showsthat only 3 percent of U.S. adults practice all of the “Big Four”habits that help to prevent heart disease: eating a healthy diet, getting regular physical activity, maintaining a healthy weight, andavoiding smoking. Many young people are also vulnerable. Arecent study showed that about two-thirds of teenagers already haveat least one risk factor for heart disease.\nEvery risk factor counts. Research shows that each individual risk\nfactor greatly increases the chances of developing heart disea

In [ ]:
len(chunks)

In [ ]:
chunks[600]

# Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

# Embeddings Creation

In [ ]:

import os
from dotenv import load_dotenv
load_dotenv()
os.environ['HUGGINGFACE_API_TOKEN'] = os.getenv('HUGGINGFACE_API_TOKEN', '')


In [ ]:
embeddings = SentenceTransformerEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")

<ipython-input-9-0ff5cf6e8b20>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")
/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a to

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Vector Store Creation

In [ ]:
vector_store = Chroma.from_documents(chunks, embeddings)
query = "Who is at risk of heart disease?"
search_result = vector_store.similarity_search(query)
search_result

[Document(metadata={'page': 8, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='While each risk factor increases your risk of heart disease, having'),
 Document(metadata={'page': 5, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='■As early as age 45, a man’s risk of heart disease begins to rise significantly. For a woman, risk starts to increase at age 55.\n■Fifty percent of men and 64 percent of women who die suddenlyof heart disease have no previous symptoms of the disease.1Heart Disease: Why Should You Care?Heart Disease:'),
 Document(metadata={'page': 6, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='factor greatly increases the chances of developing heart disease.Moreover, the worse a particular risk factor is, the more likely youare to develop heart disease. For example, if you have high bloodpressure, the higher it is, the greater your chances of developingheart disease, i

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k":5} )
retriever.get_relevant_documents(query)

<ipython-input-11-a42440457d0d>:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever.get_relevant_documents(query)


[Document(metadata={'page': 8, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='While each risk factor increases your risk of heart disease, having'),
 Document(metadata={'page': 5, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='■As early as age 45, a man’s risk of heart disease begins to rise significantly. For a woman, risk starts to increase at age 55.\n■Fifty percent of men and 64 percent of women who die suddenlyof heart disease have no previous symptoms of the disease.1Heart Disease: Why Should You Care?Heart Disease:'),
 Document(metadata={'page': 6, 'source': '/content/drive/MyDrive/BioMistral/Data/healthyheart.pdf'}, page_content='factor greatly increases the chances of developing heart disease.Moreover, the worse a particular risk factor is, the more likely youare to develop heart disease. For example, if you have high bloodpressure, the higher it is, the greater your chances of developingheart disease, i

# LLM Model Loading

In [ ]:
llm = LlamaCpp(
    model_path="/content/drive/MyDrive/BioMistral/Model/BioMistral-7B.Q4_K_M.gguf",
    temperature=0.2,
    max_tokens=2048,
    top_p=1
)

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /content/drive/MyDrive/BioMistral/Model/BioMistral-7B.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = hub
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 lla

# Use LLM and retriver and query , to generate final response

In [ ]:

template = """
<|context|>
You are an Medical Assistant that follows the instruction and generate the accurate response based on the query and the context provided.
Please be truthful and give direct answers.
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate


In [ ]:
prompt = ChatPromptTemplate.from_template(template)


In [ ]:
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:

response = rag_chain.invoke(query)


llama_perf_context_print:        load time =   34981.55 ms
llama_perf_context_print: prompt eval time =       0.00 ms /    73 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    39 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   63725.53 ms /   112 tokens


In [ ]:
response

'The risk of heart disease is influenced by a variety of factors, including age, family history, lifestyle habits such as smoking and physical activity, and medical conditions such as high blood pressure and diabetes.'

In [ ]:
# import sys
# while True :
#   user_input = input(f"Input query :")
#   if user_input == 'exit':
#     print("Exiting...")
#     sys.exit()
#   if user_input == "":
#     continue
#   result = rag_chain.invoke(user_input)
#   print("Answer:", result )

import sys

while True :
  try:
    user_input = input(f"Input query :")
    if user_input == 'exit':
      print("Exiting...")
      sys.exit()
    if user_input == "":
      continue
    result = rag_chain.invoke(user_input)
    print("Answer:", result )
  except KeyboardInterrupt:
    print("\nExiting...") # Print a newline before the message
    sys.exit()

Llama.generate: 54 prefix-match hit, remaining 17 prompt tokens to eval
llama_perf_context_print:        load time =   34981.55 ms
llama_perf_context_print: prompt eval time =       0.00 ms /    18 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    49 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   42651.26 ms /    67 tokens


Answer: The presentive measures are the following: 1. Wear a mask, 2. Keep social distance, 3. Avoid crowded places, 4. Wash your hands frequently, and 5. Disinfect surfaces regularly.
